In [1]:
from databricks_langchain import ChatDatabricks

# Prompts we will use
subjects_prompt = """Generate a list of 3 sub-topics that are all related to this overall topic: {topic}."""
joke_prompt = """Generate a joke about {subject}"""
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return the ID of the best one, starting 0 as the ID for the first joke. Jokes: \n\n  {jokes}"""

# LLM
model = ChatDatabricks(model="databricks-claude-sonnet-4", temperature=0) 

In [2]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel

In [17]:
model.invoke('hi')

AIMessage(content='Hello! How are you doing today? Is there anything I can help you with?', additional_kwargs={}, response_metadata={'usage': {'prompt_tokens': 8, 'completion_tokens': 20, 'total_tokens': 28}, 'prompt_tokens': 8, 'completion_tokens': 20, 'total_tokens': 28, 'model': 'us.anthropic.claude-sonnet-4-20250514-v1:0', 'model_name': 'us.anthropic.claude-sonnet-4-20250514-v1:0', 'finish_reason': 'stop'}, id='run--df0b0036-72b3-491b-a67f-32c9ce683fdc-0')

In [16]:
topic = 'animals'

class Subjects(BaseModel):
    subjects: list[str]
    
model.with_structured_output(schema=Subjects).invoke([HumanMessage(content=subjects_prompt.format(topic=topic))])

Subjects(subjects=['Animal habitats and ecosystems', 'Animal behavior and communication', 'Endangered species and conservation'])

In [3]:
from langgraph.graph import START, END, StateGraph
from typing import Annotated, TypedDict
from operator import add
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel

class Subjects(BaseModel):
    subjects: list[str]


class State(MessagesState):
    topic: str
    subjects: list[str]
    jokes: Annotated[str, add]
    best_joke: str

def topic_generator_node(state: State) -> State:
    topic = state['topic']
    response = model.with_structured_output(schema=Subjects).invoke(subjects_prompt.format(topic=topic))
    return {'subjects': response.subjects}


class JokeState(TypedDict):
    subject: str

def joke_generator_node(state: JokeState) -> State:
    subject = state['subject']
    return {'jokes': [model.invoke(joke_prompt.format(subject=subject)).content]} 


from langgraph.types import Send
def continue_to_joke_generation(state: State):
    subjects = state['subjects']
    return [Send('joke_generator', {'subject': subject}) for subject in subjects]

class JokeID(BaseModel):
    id: int

def best_joke_seletor_node(state: State) -> State:
    topic = state['topic']
    jokes = '\n\n--\n\n'.join(joke for joke in state['jokes'])
    prompt = best_joke_prompt.format(topic=topic, jokes=jokes)
    id = model.with_structured_output(JokeID).invoke(prompt)
    return {'best_joke': state['jokes'][id]}


graph = StateGraph(State)

graph.add_node('topic_generator', topic_generator_node)
graph.add_node('joke_generator', joke_generator_node)
graph.add_node('best_joke_seletor', best_joke_seletor_node)

graph.add_edge(START, 'topic_generator')
graph.add_conditional_edges('topic_generator', continue_to_joke_generation)
graph.add_edge('joke_generator', 'best_joke_seletor')
graph.add_edge('best_joke_seletor', END)

app = graph.compile()
app

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`